Task 1: Search Problem FormulationState:

-> The current (row, col) coordinate of the relief convoy navigating the 7x7 city grid.   

->Initial State: The convoy's starting depot located at cell (0, 0).   Actions: Moving Up, Down, Left, or Right to an adjacent valid cell. 

->Goal Test: Evaluating if the current coordinate equals the shelter located at cell (6, 6).   

->Path Cost: The cumulative sum of the terrain crossing costs for each cell traversed, where Clear Road costs 1, Debris costs 3, and Flooded costs 6.

Task 2: Admissible Heuristic
An admissible heuristic for this grid is the Manhattan Distance calculation: |current_row - goal_row| + |current_col - goal_col|. This heuristic is purely optimistic and admissible because it assumes every remaining step to the goal will only incur the cheapest possible terrain cost of 1, which represents a Clear Road. Because no passable cell on this grid has a cost lower than 1, the Manhattan distance will mathematically never overestimate the actual true cost of reaching the shelter, satisfying the core rule of admissibility

In [1]:
import heapq

grid_original = [
    [1, 1, 1, 0, 1, 1, 1],
    [1, 0, 1, 0, 1, 0, 1],
    [1, 0, 3, 3, 3, 0, 1],
    [1, 0, 3, 6, 3, 0, 1],
    [1, 0, 3, 3, 3, 0, 1],
    [1, 0, 0, 0, 0, 0, 1],
    [1, 1, 1, 1, 1, 1, 1],
]

# The Redesigned Grid (Modified to force Greedy to take a worse path)
# Cell (5, 3) changed from 0 to 3 to open the high-cost middle route.
grid_redesigned = [
    [1, 1, 1, 0, 1, 1, 1],
    [1, 0, 1, 0, 1, 0, 1],
    [1, 0, 3, 3, 3, 0, 1],
    [1, 0, 3, 6, 3, 0, 1],
    [1, 0, 3, 3, 3, 0, 1],
    [1, 0, 0, 3, 0, 0, 1],
    [1, 1, 1, 1, 1, 1, 1],
]

def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def get_neighbors(state, grid):
    r, c = state
    neighbors = []
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)] # Up, Down, Left, Right
    rows, cols = len(grid), len(grid[0])
    
    for dr, dc in directions:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols:
            if grid[nr][nc] != 0: # 0 is Impassable
                neighbors.append((nr, nc))
    return neighbors

def greedy_search(grid, start, goal):
    frontier = []
    count = 0
    heapq.heappush(frontier, (manhattan(start, goal), count, start, [start], 0))
    explored = set()
    nodes_explored = 0

    while frontier:
        h, _, current, path, current_cost = heapq.heappop(frontier)

        if current in explored:
            continue
            
        explored.add(current)
        nodes_explored += 1

        if current == goal:
            return path, current_cost, nodes_explored

        for nxt in get_neighbors(current, grid):
            if nxt not in explored:
                count += 1
                nxt_cost = grid[nxt[0]][nxt[1]]
                heapq.heappush(frontier, (manhattan(nxt, goal), count, nxt, path + [nxt], current_cost + nxt_cost))
                
    return None, -1, nodes_explored


def a_star_search(grid, start, goal):
    frontier = []
    count = 0
    heapq.heappush(frontier, (manhattan(start, goal), count, start, [start], 0))
    explored = set()
    g_costs = {start: 0}
    nodes_explored = 0

    while frontier:
        f, _, current, path, current_cost = heapq.heappop(frontier)

        if current in explored:
            continue
            
        explored.add(current)
        nodes_explored += 1

        if current == goal:
            return path, current_cost, nodes_explored

        for nxt in get_neighbors(current, grid):
            nxt_cost = grid[nxt[0]][nxt[1]]
            new_cost = current_cost + nxt_cost

            if nxt not in g_costs or new_cost < g_costs[nxt]:
                g_costs[nxt] = new_cost
                f_nxt = new_cost + manhattan(nxt, goal)
                count += 1
                heapq.heappush(frontier, (f_nxt, count, nxt, path + [nxt], new_cost))
                
    return None, -1, nodes_explored

# Task 6: Run Algorithms
if __name__ == "__main__":
    start, goal = (0, 0), (6, 6)
    print("--- Testing on REDESIGNED Grid ---")
    
    g_path, g_cost, g_nodes = greedy_search(grid_redesigned, start, goal)
    print(f"Greedy Search:\nPath: {g_path}\nTotal Cost: {g_cost}\nNodes Explored: {g_nodes}\n")
    
    a_path, a_cost, a_nodes = a_star_search(grid_redesigned, start, goal)
    print(f"A* Search:\nPath: {a_path}\nTotal Cost: {a_cost}\nNodes Explored: {a_nodes}")

--- Testing on REDESIGNED Grid ---
Greedy Search:
Path: [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (6, 6)]
Total Cost: 12
Nodes Explored: 13

A* Search:
Path: [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (6, 6)]
Total Cost: 12
Nodes Explored: 16


Task 7: Algorithm Analysis & Environment Classification:

On the original grid provided, Greedy and A* did not return different paths because the high-cost middle section is completely blocked by impassable cells with a cost of 0, making the outer left edge the only valid path to the shelter. To force the algorithms to differ, I redesigned the grid by changing cell (5, 3) from 0 (Impassable) to 3 (Debris) to open a continuous path down the center. This redesign worked because Greedy's priority is solely focused on reducing heuristic distance; it blindly advanced straight down the center, heavily penalizing itself on high-cost terrain like the Flooded cell at (3, 3). A*, however, incorporates the accumulating actual path cost, causing it to quickly abandon the expensive center route to take the cheaper, clear left edge.
The notification that flooded cells may suddenly become impassable changes the underlying environment classification to Dynamic and Stochastic. The environment is Dynamic because terrain variables can change independently while the convoy is moving, and Stochastic because those exact transitions are unpredictable. Consequently, the planner cannot rely on a single route computed at the depot; it will need to recalculate the route dynamically whenever real-time updates reveal that a previously plotted flooded cell has become impassable